# Phone Scrolling Detector\n\nThis notebook builds a practical detector for whether a person is likely scrolling on a phone in an image. It does this in two stages:\n\n1. Train one object detector that recognizes `face`, `person`, and `mobile_phone`\n2. Convert those detections into a binary `is_scrolling` decision using person/face/phone geometry\n\n## Dataset choices\n\n- **WIDER FACE** for face bounding boxes. In `torchvision`, `WIDERFace(..., download=True)` downloads the official dataset archives and annotations.\n- **Open Images V7** for `Person` and `Mobile phone` boxes, loaded as a small targeted subset via FiftyOne.\n\n## Why this design\n\nThere is no widely-used official dataset that directly labels the action *scrolling on a phone* as a standalone class. A good practical approximation is:\n\n- detect the face\n- detect the phone\n- optionally detect the person\n- classify the image as *scrolling* when the phone is held in a typical hand-use region relative to the face/body\n\nThis is a useful baseline and easy to iterate on once you collect task-specific examples.\n

## References\n\n- WIDER FACE: https://mmlab.ie.cuhk.edu.hk/projects/WIDERFace/index.html\n- Torchvision WIDERFace dataset loader: https://docs.pytorch.org/vision/master/_modules/torchvision/datasets/widerface.html\n- Open Images download page: https://storage.googleapis.com/openimages/web/download.html\n- FiftyOne Open Images V7 dataset zoo docs: https://docs.voxel51.com/dataset_zoo/datasets/open_images_v7.html\n\nIf the WIDER FACE project page is flaky, the `torchvision.datasets.WIDERFace` loader is still a reliable programmatic path because it uses the official archive IDs plus the published annotation zip URL.\n

In [ ]:
%pip install -q ultralytics fiftyone gdown torch torchvision pyyaml tqdm pillow\n

In [ ]:
from __future__ import annotations\n\nimport math\nimport os\nimport random\nimport shutil\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport yaml\nfrom PIL import Image, ImageDraw\nfrom tqdm.auto import tqdm\n\nimport torch\nfrom torchvision.datasets import WIDERFace\n\nimport fiftyone as fo\nimport fiftyone.zoo as foz\n\nfrom ultralytics import YOLO\n\nSEED = 42\nrandom.seed(SEED)\ntorch.manual_seed(SEED)\n\n@dataclass\nclass Config:\n    project_root: Path = Path.cwd()\n    data_root: Path = Path.cwd() / "data"\n    raw_root: Path = Path.cwd() / "data" / "raw"\n    yolo_root: Path = Path.cwd() / "data" / "scrolling_yolo"\n    runs_root: Path = Path.cwd() / "runs"\n    notes_root: Path = Path.cwd() / "artifacts"\n    open_images_train_samples: int = 1200\n    open_images_val_samples: int = 250\n    epochs: int = 30\n    imgsz: int = 640\n    batch: int = 16\n    device: str = "0" if torch.cuda.is_available() else "cpu"\n\ncfg = Config()\n\nfor path in [cfg.data_root, cfg.raw_root, cfg.yolo_root, cfg.runs_root, cfg.notes_root]:\n    path.mkdir(parents=True, exist_ok=True)\n\nCLASS_TO_ID = {"face": 0, "person": 1, "mobile_phone": 2}\nID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}\n\ncfg\n

## Step 1: Download the source datasets\n\nNotes:\n\n- `WIDERFace(..., download=True)` needs `gdown` because the image archives are mirrored via Google Drive in `torchvision`.\n- Open Images is large, so we intentionally download only a detection subset containing `Person` and `Mobile phone`.\n- Start with the sample counts below, then scale up once the pipeline works locally.\n

In [ ]:
wider_train = WIDERFace(root=cfg.raw_root, split="train", download=True)\nwider_val = WIDERFace(root=cfg.raw_root, split="val", download=True)\n\noi_train = foz.load_zoo_dataset(\n    "open-images-v7",\n    split="train",\n    label_types=["detections"],\n    classes=["Person", "Mobile phone"],\n    max_samples=cfg.open_images_train_samples,\n    shuffle=True,\n    seed=SEED,\n    dataset_name="open-images-v7-scroll-train",\n)\n\noi_val = foz.load_zoo_dataset(\n    "open-images-v7",\n    split="validation",\n    label_types=["detections"],\n    classes=["Person", "Mobile phone"],\n    max_samples=cfg.open_images_val_samples,\n    shuffle=True,\n    seed=SEED,\n    dataset_name="open-images-v7-scroll-val",\n)\n\nprint(f"WIDER FACE train images: {len(wider_train)}")\nprint(f"WIDER FACE val images:   {len(wider_val)}")\nprint(f"Open Images train samples: {len(oi_train)}")\nprint(f"Open Images val samples:   {len(oi_val)}")\n

## Step 2: Convert both datasets into one YOLO detection dataset\n\nThe combined detector will learn:\n\n- `face` from WIDER FACE\n- `person` and `mobile_phone` from Open Images\n\nThis cell writes:\n\n- `data/scrolling_yolo/images/train|val`\n- `data/scrolling_yolo/labels/train|val`\n- `data/scrolling_yolo/scrolling.yaml`\n

In [ ]:
def reset_split_dirs(root: Path, split: str) -> tuple[Path, Path]:\n    images_dir = root / "images" / split\n    labels_dir = root / "labels" / split\n    if images_dir.exists():\n        shutil.rmtree(images_dir)\n    if labels_dir.exists():\n        shutil.rmtree(labels_dir)\n    images_dir.mkdir(parents=True, exist_ok=True)\n    labels_dir.mkdir(parents=True, exist_ok=True)\n    return images_dir, labels_dir\n\ndef xywh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int) -> tuple[float, float, float, float] | None:\n    if w <= 1 or h <= 1:\n        return None\n    x = max(0.0, min(x, img_w - 1))\n    y = max(0.0, min(y, img_h - 1))\n    w = max(1.0, min(w, img_w - x))\n    h = max(1.0, min(h, img_h - y))\n    x_center = (x + w / 2.0) / img_w\n    y_center = (y + h / 2.0) / img_h\n    return x_center, y_center, w / img_w, h / img_h\n\ndef save_yolo_label(label_path: Path, rows: list[str]) -> None:\n    label_path.write_text("\\n".join(rows) + ("\\n" if rows else ""), encoding="utf-8")\n\ndef widerface_to_yolo(dataset: WIDERFace, split: str, images_dir: Path, labels_dir: Path, limit: int | None = None) -> int:\n    written = 0\n    iterable = dataset.img_info if limit is None else dataset.img_info[:limit]\n    for idx, info in enumerate(tqdm(iterable, desc=f"WIDERFace -> {split}")):\n        image_path = Path(info["img_path"])\n        with Image.open(image_path) as im:\n            img_w, img_h = im.size\n        boxes = info["annotations"]["bbox"].tolist()\n        rows = []\n        for x, y, w, h in boxes:\n            converted = xywh_to_yolo(float(x), float(y), float(w), float(h), img_w, img_h)\n            if converted is None:\n                continue\n            xc, yc, wn, hn = converted\n            rows.append(f"{CLASS_TO_ID['face']} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")\n        if not rows:\n            continue\n        target_name = f"wider_{split}_{idx:06d}{image_path.suffix.lower()}"\n        shutil.copy2(image_path, images_dir / target_name)\n        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)\n        written += 1\n    return written\n\ndef find_fiftyone_detection_field(dataset: fo.Dataset) -> str:\n    for field_name, field in dataset.get_field_schema().items():\n        document_type = getattr(field, "document_type", None)\n        if document_type is not None and document_type.__name__ == "Detections":\n            return field_name\n    raise RuntimeError("No FiftyOne Detections field found in dataset schema")\n\ndef openimages_to_yolo(dataset: fo.Dataset, split: str, images_dir: Path, labels_dir: Path) -> int:\n    det_field = find_fiftyone_detection_field(dataset)\n    class_map = {"Person": CLASS_TO_ID["person"], "Mobile phone": CLASS_TO_ID["mobile_phone"]}\n    written = 0\n    for idx, sample in enumerate(tqdm(dataset.iter_samples(progress=True, autosave=False), desc=f"OpenImages -> {split}")):\n        detections = getattr(sample, det_field, None)\n        if detections is None or not detections.detections:\n            continue\n        image_path = Path(sample.filepath)\n        rows = []\n        for det in detections.detections:\n            if det.label not in class_map:\n                continue\n            x, y, w, h = det.bounding_box\n            xc = x + (w / 2.0)\n            yc = y + (h / 2.0)\n            rows.append(f"{class_map[det.label]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")\n        if not rows:\n            continue\n        target_name = f"oi_{split}_{idx:06d}{image_path.suffix.lower()}"\n        shutil.copy2(image_path, images_dir / target_name)\n        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)\n        written += 1\n    return written\n\ntrain_images, train_labels = reset_split_dirs(cfg.yolo_root, "train")\nval_images, val_labels = reset_split_dirs(cfg.yolo_root, "val")\n\nwider_train_written = widerface_to_yolo(wider_train, "train", train_images, train_labels)\nwider_val_written = widerface_to_yolo(wider_val, "val", val_images, val_labels)\noi_train_written = openimages_to_yolo(oi_train, "train", train_images, train_labels)\noi_val_written = openimages_to_yolo(oi_val, "val", val_images, val_labels)\n\nyaml_path = cfg.yolo_root / "scrolling.yaml"\nyaml_payload = {\n    "path": str(cfg.yolo_root.resolve()),\n    "train": "images/train",\n    "val": "images/val",\n    "names": ["face", "person", "mobile_phone"],\n}\nyaml_path.write_text(yaml.safe_dump(yaml_payload, sort_keys=False), encoding="utf-8")\n\nprint({\n    "wider_train_written": wider_train_written,\n    "wider_val_written": wider_val_written,\n    "oi_train_written": oi_train_written,\n    "oi_val_written": oi_val_written,\n    "yaml": str(yaml_path),\n})\n

## Step 3: Train the detector\n\nA small YOLO model is a good baseline here. Swap `yolo11n.pt` for a larger checkpoint if you have more compute.\n

In [ ]:
detector = YOLO("yolo11n.pt")\n\ntrain_results = detector.train(\n    data=str(yaml_path),\n    epochs=cfg.epochs,\n    imgsz=cfg.imgsz,\n    batch=cfg.batch,\n    device=cfg.device,\n    project=str(cfg.runs_root),\n    name="phone_scrolling_detector",\n    exist_ok=True,\n    pretrained=True,\n    workers=2,\n)\n\nbest_weights = Path(detector.trainer.best)\nbest_weights\n

## Step 4: Convert detections into a binary `is_scrolling` signal\n\nHeuristic used here:\n\n- there is at least one detected face\n- there is at least one detected phone\n- ideally both belong to the same person box\n- the phone is in front of the torso and below the face center\n- the phone is not too far horizontally from the face\n\nThis is intentionally simple. Once you collect labeled positive and negative examples, you can replace this rule block with a learned classifier.\n

In [ ]:
trained_detector = YOLO(str(best_weights))\n\ndef xyxy_center(box: list[float]) -> tuple[float, float]:\n    x1, y1, x2, y2 = box\n    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)\n\ndef box_size(box: list[float]) -> tuple[float, float]:\n    x1, y1, x2, y2 = box\n    return (x2 - x1, y2 - y1)\n\ndef contains_point(box: list[float], point: tuple[float, float]) -> bool:\n    x1, y1, x2, y2 = box\n    px, py = point\n    return x1 <= px <= x2 and y1 <= py <= y2\n\ndef to_named_detections(result, conf: float = 0.25) -> dict[str, list[dict]]:\n    output = {"face": [], "person": [], "mobile_phone": []}\n    names = result.names\n    boxes = result.boxes\n    for cls_id, score, xyxy in zip(boxes.cls.tolist(), boxes.conf.tolist(), boxes.xyxy.tolist()):\n        if score < conf:\n            continue\n        label = names[int(cls_id)]\n        output[label].append({"box": xyxy, "score": float(score)})\n    return output\n\ndef scrolling_score(named: dict[str, list[dict]]) -> tuple[float, dict]:\n    faces = named["face"]\n    phones = named["mobile_phone"]\n    persons = named["person"]\n    if not faces or not phones:\n        return 0.0, {"reason": "missing_face_or_phone"}\n\n    best_score = 0.0\n    best_meta = {"reason": "no_match"}\n\n    for face in faces:\n        face_box = face["box"]\n        face_cx, face_cy = xyxy_center(face_box)\n        face_w, face_h = box_size(face_box)\n\n        matching_people = [p for p in persons if contains_point(p["box"], (face_cx, face_cy))]\n        candidate_people = matching_people or persons or [{"box": [face_box[0] - 2.0 * face_w, face_box[1], face_box[2] + 2.0 * face_w, face_box[3] + 6.0 * face_h]}]\n\n        for person in candidate_people:\n            person_box = person["box"]\n            person_w, person_h = box_size(person_box)\n            if person_w <= 1 or person_h <= 1:\n                continue\n            for phone in phones:\n                phone_box = phone["box"]\n                phone_cx, phone_cy = xyxy_center(phone_box)\n                phone_w, phone_h = box_size(phone_box)\n\n                if not contains_point(person_box, (phone_cx, phone_cy)):\n                    continue\n\n                horizontal_distance = abs(phone_cx - face_cx) / max(person_w, 1.0)\n                vertical_offset = (phone_cy - face_cy) / max(person_h, 1.0)\n                phone_scale = max(phone_w, phone_h) / max(person_w, 1.0)\n\n                horizontal_ok = horizontal_distance <= 0.35\n                vertical_ok = 0.02 <= vertical_offset <= 0.45\n                scale_ok = 0.03 <= phone_scale <= 0.30\n\n                score = 0.0\n                score += 0.35 if horizontal_ok else max(0.0, 0.35 - horizontal_distance)\n                score += 0.35 if vertical_ok else max(0.0, 0.35 - abs(vertical_offset - 0.18))\n                score += 0.20 if scale_ok else max(0.0, 0.20 - abs(phone_scale - 0.10))\n                score += 0.10 * min(face["score"], phone["score"])\n\n                if score > best_score:\n                    best_score = float(min(score, 1.0))\n                    best_meta = {\n                        "face_box": face_box,\n                        "phone_box": phone_box,\n                        "person_box": person_box,\n                        "horizontal_distance": horizontal_distance,\n                        "vertical_offset": vertical_offset,\n                        "phone_scale": phone_scale,\n                    }\n    return best_score, best_meta\n\ndef predict_scrolling(image_path: str | Path, det_conf: float = 0.25, scroll_threshold: float = 0.60):\n    result = trained_detector.predict(source=str(image_path), conf=det_conf, verbose=False)[0]\n    named = to_named_detections(result, conf=det_conf)\n    score, meta = scrolling_score(named)\n    return {\n        "image_path": str(image_path),\n        "is_scrolling": score >= scroll_threshold,\n        "scroll_score": round(score, 4),\n        "detections": named,\n        "meta": meta,\n    }\n

## Step 5: Try it on an image\n\nPut a test image path in `TEST_IMAGE`. If you have a webcam or short clips, you can run this function frame-by-frame.\n

In [ ]:
TEST_IMAGE = None\n\nif TEST_IMAGE is not None:\n    prediction = predict_scrolling(TEST_IMAGE)\n    print(prediction["is_scrolling"], prediction["scroll_score"])\n    print(prediction["meta"])\nelse:\n    print("Set TEST_IMAGE to a local image path before running this cell.")\n

In [ ]:
def draw_prediction(image_path: str | Path, prediction: dict) -> Image.Image:\n    im = Image.open(image_path).convert("RGB")\n    draw = ImageDraw.Draw(im)\n    color_map = {"face": "lime", "person": "cyan", "mobile_phone": "yellow"}\n    for label, detections in prediction["detections"].items():\n        for det in detections:\n            x1, y1, x2, y2 = det["box"]\n            draw.rectangle([x1, y1, x2, y2], outline=color_map[label], width=3)\n            draw.text((x1 + 4, max(0, y1 - 14)), f"{label}:{det['score']:.2f}", fill=color_map[label])\n    draw.text((12, 12), f"scrolling={prediction['is_scrolling']} score={prediction['scroll_score']:.2f}", fill="white")\n    return im\n\nif TEST_IMAGE is not None:\n    draw_prediction(TEST_IMAGE, prediction)\n

## Next improvements\n\nStrong directions if you want better accuracy:\n\n- collect a small labeled dataset with true `scrolling` / `not_scrolling` examples from your target camera angle\n- add a face landmark or gaze model so you can verify the user is looking downward toward the phone\n- run the detector on video and require the phone to remain present for several consecutive frames\n- replace the hand-written rule with a shallow learned classifier trained on detection geometry features\n